In [ ]:
import numpy as np
import pandas as pd
from sklearn import tree
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet,LogisticRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import PolynomialFeatures
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import LabelEncoder,OneHotEncoder
from sklearn.preprocessing import MinMaxScaler,StandardScaler
from sklearn.tree import DecisionTreeClassifier,plot_tree
from xgboost import XGBClassifier
from xgboost import XGBRegressor
from sklearn.naive_bayes import CategoricalNB #categorical handle discrete value like (color,level etc)
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report,mean_squared_error
from sklearn.metrics import mean_squared_error,mean_absolute_error,r2_score
from matplotlib.pyplot import figure
from sklearn.svm import SVC
from sklearn.svm import SVR
import seaborn as sns
from sklearn.metrics import confusion_matrix
from sklearn.multioutput import MultiOutputClassifier
import warnings
warnings.filterwarnings('ignore')

In [ ]:
df = pd.read_csv('/content/Rand of ALL Cross Dataset with URLs.csv', encoding='utf-8')
df.sample(5)

,url,headline,Language,Text,Translated_Text,Domain,Domain_ID,Translated_Domain,Translated_Domain_ID,Sub-Domain,Sub-Domain_ID,Translated_Sub-Domain,Translated_Sub-Domain_ID
1828,https://www.prothomalo.com/entertainment/world...,যৌন হয়রানির বিচার চাইতে গিয়ে যেভাবে পরিচালক হয়...,bn,নোরিয়ুকি ইয়ামাগুচি। জাপানের প্রভাবশালী এক সাংব...,Noriyuki Yamaguchi. An influential journalist ...,বিনোদন,2,Entertainment,2,সিনেমা,8,Movie,8
394,https://www.thedailystar.net/entertainment/mus...,Runa Laila’s song inspires endangered Indian t...,en,"In Totopara, a small village on the Bhutan bor...",পশ্চিমবঙ্গের আলিপুরদুয়ার জেলার ভুটান সীমান্তে...,Entertainment,2,বিনোদন,2,Music,7,সঙ্গীত,7
4281,https://www.prothomalo.com/entertainment/world...,"কে–পপ তারকার আত্মহত্যার চেষ্টা, এখনো জ্ঞান ফেরেনি",bn,আত্মহত্যার চেষ্টার পর কে–পপ তারকা লি আরিয়মকে হ...,K-pop star Lee Ariem has been hospitalized aft...,বিনোদন,2,Entertainment,2,সিনেমা,8,Movie,8
4139,NaN,NaN,en,"Athletes are often cast as entertainers, their...",ক্রীড়াবিদদের প্রায়শই বিনোদনদাতা হিসেবে দেখা ...,Sports,0,খেলাধুলা,0,Tennis,2,টেনিস,2
2932,https://www.prothomalo.com/bangladesh/crime/fo...,বাড্ডায় যুবকের গুলিবিদ্ধ মরদেহ উদ্ধার,bn,রাজধানীর মধ্য বাড্ডা এলাকার কমিশনার গলির একটি ...,Police have recovered the bullet-riddled body ...,অপরাধ,1,Crime,1,সহিংস অপরাধ,3,Violent crime,3


In [ ]:
!pip install transformers
!pip install datasets

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np

# Prepare nlp_df
nlp_df = df.copy()

# 1. Directly use 'Domain_ID' as our numerical target
y_domain_numerical = nlp_df['Domain_ID'].astype(int)

# 2. Define X_domain as the 'Text' column from nlp_df
X_domain = nlp_df['Text'].str.strip()

# 3. Split into 80% Train and 20% Temporary (Validation + Test)
X_train_domain, X_temp_domain, y_train_domain, y_temp_domain = train_test_split(
    X_domain, y_domain_numerical, test_size=0.2, random_state=42, stratify=y_domain_numerical
)

# 4. Split the 20% temporary set equally (50% each) to get exactly 10% Validation and 10% Test
X_val_domain, X_test_domain, y_val_domain, y_test_domain = train_test_split(
    X_temp_domain, y_temp_domain, test_size=0.5, random_state=42, stratify=y_temp_domain
)

# 5. Print the shapes of the resulting datasets
print(f"Shape of X_train_domain: {X_train_domain.shape} (80%)")
print(f"Shape of y_train_domain: {y_train_domain.shape}")
print(f"Shape of X_val_domain: {X_val_domain.shape} (10%)")
print(f"Shape of y_val_domain: {y_val_domain.shape}")
print(f"Shape of X_test_domain: {X_test_domain.shape} (10%)")
print(f"Shape of y_test_domain: {y_test_domain.shape}")

# 6. Print sample numerical labels from y_train_domain
print(f"\nSample of y_train_domain: {y_train_domain[:5].tolist()}")

Shape of X_train_domain: (8745,) (80%)
Shape of y_train_domain: (8745,)
Shape of X_val_domain: (1093,) (10%)
Shape of y_val_domain: (1093,)
Shape of X_test_domain: (1094,) (10%)
Shape of y_test_domain: (1094,)

Sample of y_train_domain: [4, 1, 3, 2, 4]


In [ ]:
from transformers import AutoTokenizer
from datasets import Dataset, DatasetDict

# Ensure X_train_domain, y_train_domain, X_val_domain, y_val_domain, X_test_domain, y_test_domain are defined.
# These should have been set up by cell '1595d2c3' and are available in the environment.

# Define MAX_LENGTH for tokenization (reusing the value from previous cells)
MAX_LENGTH_DOMAIN = 256 # A common length for BERT models

# 1. Define the model name for xlm-roberta
xlm_roberta_domain_model_name = 'xlm-roberta-base'

# 2. Load the tokenizer for the xlm-roberta model
xlm_roberta_domain_tokenizer = AutoTokenizer.from_pretrained(xlm_roberta_domain_model_name)

# 3. Create a DatasetDict containing 'train', 'validation', and 'test' datasets
dataset_dict_domain_xlm_roberta = DatasetDict({
    'train': Dataset.from_dict({
        'text': X_train_domain.tolist(),
        'label': y_train_domain.tolist()
    }),
    'validation': Dataset.from_dict({
        'text': X_val_domain.tolist(),
        'label': y_val_domain.tolist()
    }),
    'test': Dataset.from_dict({
        'text': X_test_domain.tolist(),
        'label': y_test_domain.tolist()
    })
})

# 4. Create a tokenization function
def tokenize_function_xlm_roberta_domain(examples):
    tokenized_inputs = xlm_roberta_domain_tokenizer(
        examples['text'],
        truncation=True,
        padding='max_length',
        max_length=MAX_LENGTH_DOMAIN
    )
    tokenized_inputs['labels'] = examples['label'] # Ensure 'labels' key is present
    return tokenized_inputs

# 5. Apply this tokenization function to all datasets
processed_datasets_xlm_roberta_domain = dataset_dict_domain_xlm_roberta.map(tokenize_function_xlm_roberta_domain, batched=True)

# 6. Set the format of the processed datasets to 'torch'
processed_datasets_xlm_roberta_domain.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

print("XLM-RoBERTa Tokenizer loaded and 'Domain' data processed.")
print(f"Sample of processed_datasets_xlm_roberta_domain['train'][0]: {processed_datasets_xlm_roberta_domain['train'][0]}")
print(f"Sample of processed_datasets_xlm_roberta_domain['validation'][0]: {processed_datasets_xlm_roberta_domain['validation'][0]}")
print(f"Sample of processed_datasets_xlm_roberta_domain['test'][0]: {processed_datasets_xlm_roberta_domain['test'][0]}")

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

Map:   0%|          | 0/8745 [00:00<?, ? examples/s]

Map:   0%|          | 0/1093 [00:00<?, ? examples/s]

Map:   0%|          | 0/1094 [00:00<?, ? examples/s]

XLM-RoBERTa Tokenizer loaded and 'Domain' data processed.
Sample of processed_datasets_xlm_roberta_domain['train'][0]: {'input_ids': tensor([     0,  73078, 122305, 104216,   9618,  69057,   5507,  63094, 209423,
          4557,      6, 112542,  13327,  44672,  66106, 116605,   2763, 176842,
         32368, 110362, 213065,      6, 112542,  13327,  44672,  66106, 116605,
          2763, 169720,  42579,  72999,  28746, 154485,    125,   6386, 170478,
         72229,      6, 104694,  86443,  87084, 206912,  61705,    896,   4691,
        125902,    999, 170812,  77162,  41450, 170045,   4557,    125,  25508,
         97457, 187909, 200966,   2784,  95085,  16550,   4979, 221461,   8339,
         19368, 122948,    956,   4577, 170812,  77162,   7185,   9681,    125,
        136729,      6, 222056, 148264, 161277, 170045,  64123,   4979,  10062,
         34510,  13327,   2784,      6,  17891, 178894, 184515, 119977,    125,
          4577,  20895, 170045,    896,  52431, 215840,    999,  57

##Domain Parameters

In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer
import torch
import gc

# --- Start: Ensure nlp_df and domain classification data are prepared ---
nlp_df = df.copy()

y_domain_numerical = nlp_df['Domain_ID'].astype(int)
X_domain = nlp_df['Text'].str.strip() #+ ' ' + nlp_df['Translated_Text'] # Handle potential NaN values in text

# Split X_domain and y_domain_numerical
X_train_domain, X_temp_domain, y_train_domain, y_temp_domain = train_test_split(
    X_domain, y_domain_numerical, test_size=0.2, random_state=42, stratify=y_domain_numerical
)

X_val_domain, X_test_domain, y_val_domain, y_test_domain = train_test_split(
    X_temp_domain, y_temp_domain, test_size=0.5, random_state=42, stratify=y_temp_domain
)

MAX_LENGTH_DOMAIN = 256
xlm_roberta_domain_model_name = 'xlm-roberta-base'
xlm_roberta_domain_tokenizer = AutoTokenizer.from_pretrained(xlm_roberta_domain_model_name)

dataset_dict_domain_xlm_roberta = DatasetDict({
    'train': Dataset.from_dict({
        'text': X_train_domain.tolist(),
        'label': y_train_domain.tolist()
    }),
    'validation': Dataset.from_dict({
        'text': X_val_domain.tolist(),
        'label': y_val_domain.tolist()
    }),
    'test': Dataset.from_dict({
        'text': X_test_domain.tolist(),
        'label': y_test_domain.tolist()
    })
})

def tokenize_function_xlm_roberta_domain(examples):
    tokenized_inputs = xlm_roberta_domain_tokenizer(
        examples['text'],
        truncation=True,
        padding='max_length',
        max_length=MAX_LENGTH_DOMAIN
    )
    tokenized_inputs['labels'] = examples['label']
    return tokenized_inputs

processed_datasets_xlm_roberta_domain = dataset_dict_domain_xlm_roberta.map(tokenize_function_xlm_roberta_domain, batched=True)
processed_datasets_xlm_roberta_domain.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
# --- End: Ensure nlp_df and domain classification data are prepared ---

# 1. Get number of unique labels for the model's output layer for 'Domain' classification
num_labels_domain = int(y_domain_numerical.nunique())
print(f"Number of unique labels for Domain classification: {num_labels_domain}")

# --- Memory Cleanup ---
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# 2. Load the pre-trained XLM-RoBERTa model for sequence classification
xlm_roberta_domain_model = AutoModelForSequenceClassification.from_pretrained(
    xlm_roberta_domain_model_name,
    num_labels=num_labels_domain
)
print(f"Model '{xlm_roberta_domain_model_name}' loaded for Domain classification.")

# 3. Define evaluation metrics
def compute_metrics(eval_pred):
    labels = eval_pred.label_ids
    preds = eval_pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='macro', zero_division=0)
    acc = accuracy_score(labels, preds)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

# Check if CUDA (GPU) is available for training
use_cuda = torch.cuda.is_available()
print(f"Using CUDA (GPU) for training: {use_cuda}")

# 4. Configure training arguments for Domain classification with optimized memory parameters
training_args_xlm_roberta_domain = TrainingArguments(
    output_dir='./results_xlm_roberta_domain',
    num_train_epochs=5,
    per_device_train_batch_size=16,         # Increased for L4 GPU
    per_device_eval_batch_size=32,
    warmup_steps=500,
    weight_decay=0.01,
    learning_rate=2e-5,                     # Added learning rate
    # logging_dir='./logs_xlm_roberta_domain',
    logging_steps=10,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    fp16=use_cuda,                           # Enable mixed precision only if GPU is available
    report_to='none'
)

# 5. Initialize the Trainer for Domain classification
trainer_xlm_roberta_domain = Trainer(
    model=xlm_roberta_domain_model,
    args=training_args_xlm_roberta_domain,
    train_dataset=processed_datasets_xlm_roberta_domain['train'],
    eval_dataset=processed_datasets_xlm_roberta_domain['validation'],
    compute_metrics=compute_metrics
)

print("Trainer initialized for XLM-RoBERTa Domain classification.")

# 6. Train the model
trainer_xlm_roberta_domain.train()

print("XLM-RoBERTa Domain classification training complete. Evaluating model on the test set.")

# 7. Evaluate the model on the test set
results_xlm_roberta_domain = trainer_xlm_roberta_domain.evaluate(processed_datasets_xlm_roberta_domain['test'])

print("XLM-RoBERTa Domain classification Test set evaluation results:")
for key, value in results_xlm_roberta_domain.items():
    print(f"  {key}: {value:.4f}")

# 8. Create a pandas DataFrame for XLM-RoBERTa's Domain classification metrics
xlm_roberta_domain_metrics_data = {
    'Model': ['XLM-RoBERTa(Domain_Flat)'],
    'Accuracy': [results_xlm_roberta_domain['eval_accuracy']],
    'Precision (Macro Avg)': [results_xlm_roberta_domain['eval_precision']],
    'Recall (Macro Avg)': [results_xlm_roberta_domain['eval_recall']],
    'F1-Score (Macro Avg)': [results_xlm_roberta_domain['eval_f1']]
}
xlm_roberta_domain_metrics_df = pd.DataFrame(xlm_roberta_domain_metrics_data)

# 9. Display the xlm_roberta_domain_metrics_df DataFrame
display(xlm_roberta_domain_metrics_df)

Map:   0%|          | 0/8745 [00:00<?, ? examples/s]

Map:   0%|          | 0/1093 [00:00<?, ? examples/s]

Map:   0%|          | 0/1094 [00:00<?, ? examples/s]

Number of unique labels for Domain classification: 5


model.safetensors: reconstructing file:   0%|          |  0.00B / 1.12GB            

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

XLM-RoBERTa Domain classification training complete. Evaluating model on the test set.


Training Loss,Validation Loss,Epoch,Accuracy,F1,Precision,Recall
0.091209,0.219837,5,0.953382,0.952851,0.952234,0.954269


XLM-RoBERTa Domain classification Test set evaluation results:
  eval_loss: 0.2198
  eval_accuracy: 0.9534
  eval_f1: 0.9529
  eval_precision: 0.9522
  eval_recall: 0.9543


,Model,Accuracy,Precision (Macro Avg),Recall (Macro Avg),F1-Score (Macro Avg)
0,XLM-RoBERTa(Domain_Flat),0.953382,0.952234,0.954269,0.952851


In [ ]:
# import torch
# import numpy as np

# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# # Ensure mapping dictionaries are defined
# domain_id_to_name = nlp_df.set_index('Domain_ID')['Domain'].drop_duplicates().to_dict()
# sub_domain_id_to_name = nlp_df.set_index('Sub-Domain_ID')['Sub-Domain'].drop_duplicates().to_dict()

# xlm_roberta_domain_model.to(device)
# xlm_roberta_domain_model.eval()

# predicted_domains = []

# for _, row in nlp_df.iterrows():

#     text = str(row['Text']) if pd.notna(row['Text']) else ''
#     translated_text = str(row['Translated_Text']) if pd.notna(row['Translated_Text']) else ''

#     combined_text = text + ' ' + translated_text

#     inputs = xlm_roberta_domain_tokenizer(
#         combined_text,
#         truncation=True,
#         padding='max_length',
#         max_length=MAX_LENGTH_DOMAIN,
#         return_tensors='pt'
#     )

#     inputs = {k: v.to(device) for k, v in inputs.items()}

#     with torch.no_grad():
#         outputs = xlm_roberta_domain_model(**inputs)

#     predicted_id = torch.argmax(outputs.logits, dim=-1).item()

#     predicted_domain = domain_id_to_name.get(
#         predicted_id,
#         "Unknown Domain"
#     )

#     predicted_domains.append(predicted_domain)

# nlp_df['Predicted_Domain'] = predicted_domains

#Sub-Dmain

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np

# 1. Directly use 'Sub-Domain_ID' as our numerical target
y_sub_domain_numerical = nlp_df['Sub-Domain_ID'].astype(int)

# 2. Define X_sub_domain as the concatenation of 'Domain' and 'Text'
X_sub_domain = nlp_df['Text'].fillna('').astype(str) #(nlp_df['Domain'].fillna('').astype(str) + " " +

# 3. Split into 80% Train and 20% Temporary (Validation + Test)
X_train_sub_domain, X_temp_sub_domain, y_train_sub_domain, y_temp_sub_domain = train_test_split(
    X_sub_domain, y_sub_domain_numerical, test_size=0.2, random_state=42, stratify=y_sub_domain_numerical
)

# 4. Split the 20% temporary set equally (50% each) to get exactly 10% Validation and 10% Test
X_val_sub_domain, X_test_sub_domain, y_val_sub_domain, y_test_sub_domain = train_test_split(
    X_temp_sub_domain, y_temp_sub_domain, test_size=0.5, random_state=42, stratify=y_temp_sub_domain
)

# 5. Print the shapes of the resulting datasets
print(f"Shape of X_train_sub_domain: {X_train_sub_domain.shape} (80%)")
print(f"Shape of y_train_sub_domain: {y_train_sub_domain.shape}")
print(f"Shape of X_val_sub_domain: {X_val_sub_domain.shape} (10%)")
print(f"Shape of y_val_sub_domain: {y_val_sub_domain.shape}")
print(f"Shape of X_test_sub_domain: {X_test_sub_domain.shape} (10%)")
print(f"Shape of y_test_sub_domain: {y_test_sub_domain.shape}")

# 6. Print sample numerical labels from y_train_sub_domain
print(f"\nSample of y_train_sub_domain: {y_train_sub_domain[:5].tolist()}")

Shape of X_train_sub_domain: (8745,) (80%)
Shape of y_train_sub_domain: (8745,)
Shape of X_val_sub_domain: (1093,) (10%)
Shape of y_val_sub_domain: (1093,)
Shape of X_test_sub_domain: (1094,) (10%)
Shape of y_test_sub_domain: (1094,)

Sample of y_train_sub_domain: [10, 11, 1, 6, 9]


In [ ]:
from transformers import AutoTokenizer
from datasets import Dataset, DatasetDict

# Ensure X_train_sub_domain, y_train_sub_domain, etc. are available from previous steps
# (specifically cell 'f3f3c2dc')

# Define MAX_LENGTH for tokenization (reusing the value from Domain classification)
MAX_LENGTH_SUB_DOMAIN = 512

# Prepare dataset_dict_sub_domain (if not already in scope from cell 72dbd092)
# Assuming X_train_sub_domain, y_train_sub_domain, etc. are populated.
dataset_dict_sub_domain_xlm_roberta = DatasetDict({
    'train': Dataset.from_dict({
        'text': X_train_sub_domain.tolist(),
        'label': y_train_sub_domain.tolist()
    }),
    'validation': Dataset.from_dict({
        'text': X_val_sub_domain.tolist(),
        'label': y_val_sub_domain.tolist()
    }),
    'test': Dataset.from_dict({
        'text': X_test_sub_domain.tolist(),
        'label': y_test_sub_domain.tolist()
    })
})

# 1. Define the model name for xlm-roberta
xlm_roberta_sub_domain_model_name = 'xlm-roberta-base'

# 2. Load the tokenizer for the xlm-roberta model
xlm_roberta_sub_domain_tokenizer = AutoTokenizer.from_pretrained(xlm_roberta_sub_domain_model_name)

# 3. Create a tokenization function
def tokenize_function_xlm_roberta_sub_domain(examples):
    tokenized_inputs = xlm_roberta_sub_domain_tokenizer(
        examples['text'],
        truncation=True,
        padding='max_length',
        max_length=MAX_LENGTH_SUB_DOMAIN
    )
    tokenized_inputs['labels'] = examples['label'] # Ensure 'labels' key is present
    return tokenized_inputs

# 4. Apply this tokenization function to all datasets
processed_datasets_xlm_roberta_sub_domain = dataset_dict_sub_domain_xlm_roberta.map(tokenize_function_xlm_roberta_sub_domain, batched=True)

# 5. Set the format of the processed datasets to 'torch'
processed_datasets_xlm_roberta_sub_domain.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

print("XLM-RoBERTa Tokenizer loaded and 'Sub-Domain' data processed.")
print(f"Sample of processed_datasets_xlm_roberta_sub_domain['train'][0]: {processed_datasets_xlm_roberta_sub_domain['train'][0]}")
print(f"Sample of processed_datasets_xlm_roberta_sub_domain['validation'][0]: {processed_datasets_xlm_roberta_sub_domain['validation'][0]}")
print(f"Sample of processed_datasets_xlm_roberta_sub_domain['test'][0]: {processed_datasets_xlm_roberta_sub_domain['test'][0]}")

Map:   0%|          | 0/8745 [00:00<?, ? examples/s]

Map:   0%|          | 0/1093 [00:00<?, ? examples/s]

Map:   0%|          | 0/1094 [00:00<?, ? examples/s]

XLM-RoBERTa Tokenizer loaded and 'Sub-Domain' data processed.
Sample of processed_datasets_xlm_roberta_sub_domain['train'][0]: {'input_ids': tensor([     0,  18336,  16460,   6155, 176997,    125,  59474,    896,  86517,
         16576,  72786, 206207, 135361,  41968, 168949,  19039, 162522,    999,
         34647,  11291,   4876, 215512, 167762,  33019, 145720, 153634,  16908,
         20824,  86517,  16576,  72786,  19039,   7443,   6638,   6386, 176997,
           999,  16447,  38841, 196695,    125,  22540,  38084,   2145,   6638,
         52431,  50442,   2763,      9,   9115,   2763,      9,   8276,   9307,
          8377,  16926,  10066, 100564,  12468,  29881,  54546,  24029,   4691,
         24373, 135361,  20134,  20824,  24373, 125506,    999,  21431,   4876,
           999,  25326,  27369,    999,   9853, 176997,    999, 203529,  63520,
         52730,  34647,  76675,  19039,  47458,  82712,  12468,  25326,    125,
             6,  98577,  14403,  23878,  30550,  58065, 155

In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import numpy as np
import pandas as pd

# 1. Get number of unique labels for the model's output layer for 'Sub-Domain' classification
# Retrieve number of classes directly from dataset
num_labels_sub_domain = int(nlp_df['Sub-Domain_ID'].nunique())
print(f"Number of unique labels for Sub-Domain classification: {num_labels_sub_domain}")

# 2. Load the pre-trained XLM-RoBERTa model for sequence classification
xlm_roberta_sub_domain_model = AutoModelForSequenceClassification.from_pretrained(
    xlm_roberta_sub_domain_model_name,
    num_labels=num_labels_sub_domain
)
print(f"Model '{xlm_roberta_sub_domain_model_name}' loaded for Sub-Domain classification.")

# 3. Define evaluation metrics
def compute_metrics(eval_pred):
    labels = eval_pred.label_ids
    preds = eval_pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='macro', zero_division=0)
    acc = accuracy_score(labels, preds)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

# 4. Configure training arguments for Sub-Domain classification
training_args_xlm_roberta_sub_domain = TrainingArguments(
    output_dir='./results_xlm_roberta_sub_domain',
    num_train_epochs=10,
    per_device_train_batch_size=16,         # Increased for L4 GPU
    per_device_eval_batch_size=32,
    warmup_steps=500,
    weight_decay=0.01,
    learning_rate=2e-5,                     # Added learning rate
    # logging_dir='./logs_xlm_roberta_sub_domain',
    logging_steps=10,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    fp16=use_cuda,                           # Enable mixed precision only if GPU is available
    report_to='none'
)

# 5. Initialize the Trainer for Sub-Domain classification
trainer_xlm_roberta_sub_domain = Trainer(
    model=xlm_roberta_sub_domain_model,
    args=training_args_xlm_roberta_sub_domain,
    train_dataset=processed_datasets_xlm_roberta_sub_domain['train'],
    eval_dataset=processed_datasets_xlm_roberta_sub_domain['validation'],
    compute_metrics=compute_metrics
)

print("Trainer initialized for XLM-RoBERTa Sub-Domain classification.")

# 6. Train the model
trainer_xlm_roberta_sub_domain.train()

print("XLM-RoBERTa Sub-Domain classification training complete. Evaluating model on the test set.")

# 7. Evaluate the model on the test set
results_xlm_roberta_sub_domain = trainer_xlm_roberta_sub_domain.evaluate(processed_datasets_xlm_roberta_sub_domain['test'])

print("XLM-RoBERTa Sub-Domain classification Test set evaluation results:")
for key, value in results_xlm_roberta_sub_domain.items():
    print(f"  {key}: {value:.4f}")

# 8. Create a pandas DataFrame for XLM-RoBERTa's Sub-Domain classification metrics
xlm_roberta_sub_domain_metrics_data = {
    'Model': ['XLM-RoBERTa (Sub-Domain_Flat)'],
    'Accuracy': [results_xlm_roberta_sub_domain['eval_accuracy']],
    'Precision (Macro Avg)': [results_xlm_roberta_sub_domain['eval_precision']],
    'Recall (Macro Avg)': [results_xlm_roberta_sub_domain['eval_recall']],
    'F1-Score (Macro Avg)': [results_xlm_roberta_sub_domain['eval_f1']]
}
xlm_roberta_sub_domain_metrics_df = pd.DataFrame(xlm_roberta_sub_domain_metrics_data)

# 9. Display the xlm_roberta_sub_domain_metrics_df DataFrame
display(xlm_roberta_sub_domain_metrics_df)

Number of unique labels for Sub-Domain classification: 15


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model 'xlm-roberta-base' loaded for Sub-Domain classification.
Trainer initialized for XLM-RoBERTa Sub-Domain classification.


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.773260,0.752368,0.765782,0.761458,0.777276,0.764662
2,0.675165,0.571553,0.814273,0.813767,0.813691,0.821348
3,0.428517,0.522399,0.821592,0.824976,0.827709,0.827437
4,0.313166,0.549332,0.833486,0.835196,0.835364,0.837958
5,0.308817,0.590210,0.818847,0.818222,0.820882,0.819546
6,0.202644,0.599599,0.831656,0.826896,0.841302,0.823479
7,0.245944,0.673542,0.820677,0.821803,0.824908,0.821029
8,0.145488,0.709146,0.827996,0.830180,0.833394,0.830019
9,0.049817,0.786611,0.837145,0.835623,0.837662,0.835792
10,0.122370,0.814102,0.838060,0.836360,0.838274,0.836381


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

XLM-RoBERTa Sub-Domain classification training complete. Evaluating model on the test set.


Training Loss,Validation Loss,Epoch,Accuracy,F1,Precision,Recall
0.122370,0.944032,10,0.813528,0.810321,0.810599,0.812077


XLM-RoBERTa Sub-Domain classification Test set evaluation results:
  eval_loss: 0.9440
  eval_accuracy: 0.8135
  eval_f1: 0.8103
  eval_precision: 0.8106
  eval_recall: 0.8121


,Model,Accuracy,Precision (Macro Avg),Recall (Macro Avg),F1-Score (Macro Avg)
0,XLM-RoBERTa (Sub-Domain_Flat),0.813528,0.810599,0.812077,0.810321


## Sub-domain Train Loss

In [ ]:
import plotly.graph_objects as go
import pandas as pd

def plot_interactive_loss(trainer, title):
    # Extract log history
    history = trainer.state.log_history

    train_loss = [entry for entry in history if 'loss' in entry and 'epoch' in entry]
    val_loss = [entry for entry in history if 'eval_loss' in entry and 'epoch' in entry]

    df_train = pd.DataFrame(train_loss)
    df_val = pd.DataFrame(val_loss)

    fig = go.Figure()

    # Add Training Loss Trace
    if not df_train.empty:
        fig.add_trace(go.Scatter(
            x=df_train['epoch'],
            y=df_train['loss'],
            mode='lines+markers',
            name='Training Loss',
            line=dict(color='blue')
        ))

    # Add Validation Loss Trace
    if not df_val.empty:
        fig.add_trace(go.Scatter(
            x=df_val['epoch'],
            y=df_val['eval_loss'],
            mode='lines+markers',
            name='Validation Loss',
            line=dict(color='red', dash='dash')
        ))

    fig.update_layout(
        title=f'Interactive Loss Curve: {title}',
        xaxis_title='Epochs',
        yaxis_title='Loss',
        template='plotly_white',
        hovermode='x unified'
    )

    fig.show()

# Generating Interactive Graph only for Sub-Domain Model
print("Generating Interactive Loss Graph for Sub-Domain Model...")
plot_interactive_loss(trainer_xlm_roberta_sub_domain, "Sub-Domain Classification (XLM-RoBERTa)")

Generating Interactive Loss Graph for Sub-Domain Model...


In [ ]:
import torch

# Create mappings from ID to Name for Domain and Sub-Domain
domain_id_to_name = nlp_df[['Domain_ID', 'Domain']].drop_duplicates().set_index('Domain_ID')['Domain'].to_dict()
sub_domain_id_to_name = nlp_df[['Sub-Domain_ID', 'Sub-Domain']].drop_duplicates().set_index('Sub-Domain_ID')['Sub-Domain'].to_dict()

print("Domain ID to Name Mapping:", domain_id_to_name)
print("Sub-Domain ID to Name Mapping:", sub_domain_id_to_name)

# Mount Google Drive
We will mount Google Drive to save the models and pickle files directly into your personal drive storage.

In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# # Define the target directory in Google Drive
# save_dir = '/content/drive/MyDrive/xlm_roberta_domain_model'
# os.makedirs(save_dir, exist_ok=True)
# print(f"Directory created or verified at: {save_dir}")

### Save the Trained Model, Tokenizer and Mappings
Now we will save the XLM-RoBERTa domain model and its tokenizer using their built-in `save_pretrained` method, and use `pickle` to export our class mapping dictionaries (`domain_id_to_name` and `sub_domain_id_to_name`).

In [ ]:
import os
import pickle

# Define directories to save models, tokenizers, and mappings
save_dir_domain = '/content/drive/MyDrive/Cross/Domain/Cross-xlm-domain'
save_dir_sub_domain = '/content/drive/MyDrive/Cross/SUb-Domain/Cross-xlm-sub-domain'

# Create directories if they don't exist
os.makedirs(save_dir_domain, exist_ok=True)
os.makedirs(save_dir_sub_domain, exist_ok=True)

# Save the Domain classification model, tokenizer, and mappings
xlm_roberta_domain_model.save_pretrained(save_dir_domain)
xlm_roberta_domain_tokenizer.save_pretrained(save_dir_domain)
with open(os.path.join(save_dir_domain, 'domain_id_to_name.pkl'), 'wb') as f:
    pickle.dump(domain_id_to_name, f)
print(f"Domain model, tokenizer, and mappings saved to: {save_dir_domain}")

# Save the Sub-Domain classification model, tokenizer, and mappings
xlm_roberta_sub_domain_model.save_pretrained(save_dir_sub_domain)
xlm_roberta_sub_domain_tokenizer.save_pretrained(save_dir_sub_domain)
with open(os.path.join(save_dir_sub_domain, 'sub_domain_id_to_name.pkl'), 'wb') as f:
    pickle.dump(sub_domain_id_to_name, f)
print(f"Sub-Domain model, tokenizer, and mappings saved to: {save_dir_sub_domain}")

#Predictions

In [ ]:
def predict_domain_and_sub_domain(text):
    # Determine the device
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Predict Domain
    inputs_domain = xlm_roberta_domain_tokenizer(text, return_tensors='pt', truncation=True, padding='max_length', max_length=MAX_LENGTH_DOMAIN)
    # Move input tensors to the same device as the model
    inputs_domain = {k: v.to(device) for k, v in inputs_domain.items()}

    # Move the model to the device if it's not already there (this should ideally be done once during model loading)
    xlm_roberta_domain_model.to(device)

    with torch.no_grad():
        outputs_domain = xlm_roberta_domain_model(**inputs_domain)
    predicted_domain_id = torch.argmax(outputs_domain.logits, dim=-1).item()
    predicted_domain = domain_id_to_name.get(predicted_domain_id, "Unknown Domain")

    # Prepare input for Sub-Domain prediction. Based on your clarification,
    # the sub-domain model was trained individually on 'Text' only (flat classification).
    # So, we only use the original text for sub-domain prediction.
    sub_domain_input_text = text # Changed from: predicted_domain + " " + text

    # Predict Sub-Domain
    inputs_sub_domain = xlm_roberta_sub_domain_tokenizer(sub_domain_input_text, return_tensors='pt', truncation=True, padding='max_length', max_length=MAX_LENGTH_SUB_DOMAIN)
    # Move input tensors to the same device as the model
    inputs_sub_domain = {k: v.to(device) for k, v in inputs_sub_domain.items()}

    # Move the model to the device if it's not already there (this should ideally be done once during model loading)
    xlm_roberta_sub_domain_model.to(device)

    with torch.no_grad():
        outputs_sub_domain = xlm_roberta_sub_domain_model(**inputs_sub_domain)
    predicted_sub_domain_id = torch.argmax(outputs_sub_domain.logits, dim=-1).item()
    predicted_sub_domain = sub_domain_id_to_name.get(predicted_sub_domain_id, "Unknown Sub-Domain")

    return predicted_domain, predicted_sub_domain

print("Prediction function 'predict_domain_and_sub_domain' defined.")

### Prediction Demonstration
Let's test the `predict_domain_and_sub_domain` function with a sample text.

In [ ]:
sample_text = '''
The Ministry of Commerce has granted permission to import 150,000 tonnes of crude salt ahead of the new production season. Officials from the Bangladesh Small and Cottage Industries Corporation (BSCIC) stated that this import was recommended due to fears of a potential shortage.
However, salt farmers are aggrieved by the Ministry of Commerce's decision. They claim there is no salt shortage in the country. Permission for imports was granted just as farmers—after months of losses—were beginning to see some profit.
According to BSCIC data, the total demand for refined salt in the country for the 2026–27 fiscal year is estimated at 2.1 million tonnes. This includes 750,000 tonnes of edible salt, 950,000 tonnes for industrial use, 37,000 tonnes for fish farming, and 306,000 tonnes for the livestock sector. The salt production season in the country typically runs from November to May.
Sarwar Hossain, head of BSCIC’s salt cell, told *Prothom Alo* that production was somewhat lower last season. Currently, there are 490,000 tonnes of salt in stock at the field level and 177,000 tonnes at mills. The country's monthly demand is approximately 220,000 tonnes; at this rate, current stocks will last until mid-December. Salt from the new season will begin arriving before then. However, production was very low during the first two months of last season. Based on that experience, the import recommendation was made to avert a potential shortage lasting 20 to 25 days.
On the other hand, senior officials from several salt mills argue that permission to import salt is granted almost every year citing fears of a shortage. Permission was granted last year as well but was later cancelled; no market crisis occurred even without the imports. They allege that permission to import crude salt is granted to benefit a specific group of mill owners. Ultimately, the import permits are sold at high prices, and the market price of salt rises.
Currently, there are 490,000 tonnes of salt in stock at the field level and 177,000 tonnes at mills. The country's monthly demand is approximately 220,000 tonnes.
AJM Siam, General Secretary of the Bangladesh Salt Farmers' Welfare Council, stated that the production cost per *maund* (approx. 37.3 kg) of salt last season was between Tk 320 and Tk 330. At the beginning of the season, farmers sold salt at a loss of Tk 40 to 50 per maund. Prices began to rise towards the end of the season; currently, with sales at Tk 380 per maund, they are finally seeing some profit. Granting import permission at this juncture would cause prices to drop.
There are a total of 251 salt mills across the country, located in Dhaka, Narayanganj, Chandpur, Chattogram, Patiya, Cox's Bazar, Khulna, and Jhalakathi. Among them, seven are modern vacuum mills: ACI Salt, Molla Salt, Triple Refinery Industries, Confidence Salt, City Group, Meghna Group of Industries, and SA Salt. These companies control 70 percent of the market for edible and industrial salt.
Inquiries reveal that, following lobbying by leaders of the Bangladesh Salt Mill Owners Association, BSCIC recommended salt imports in the middle of this year. Accordingly, the Ministry of Commerce granted permission last Monday to import 150,000 tonnes of crude salt.
Under this permission, 244 mills have been allowed to import a total of 132,500 tonnes (543 tonnes each), while the seven vacuum-process salt-producing companies have been permitted to import a total of 17,500 tonnes (2,500 tonnes each). However, it has been reported that 86 of the 244 mills did not produce any edible salt between July of last year and August of this year.
Nurul Kabir, President of the Bangladesh Salt Mill Owners Association, stated that one cannot practice discrimination when running an organization. He added that the government granted import permission to avert the risk of a crisis.
Sarwar Hossain, head of BSCIC’s salt cell, noted that in the past, many mill owners had gone to court over allocations based on production capacity—incidents that occurred twice previously. He warned that this time, if anyone sells their import permit, their registration will be cancelled.
'''
predicted_domain, predicted_sub_domain = predict_domain_and_sub_domain(sample_text)

print(f"Sample Text: {sample_text}")
print(f"Predicted Domain: {predicted_domain}")
print(f"Predicted Sub-Domain: {predicted_sub_domain}")

sample_text_2 = '''সাবেক আইনমন্ত্রী আনিসুল হকসহ ৪ জনের বিরুদ্ধে ১৭২ কোটি ২৪ লাখ ৩২ হাজার ৯৪১ টাকার ‘অবৈধ সম্পদ’ অর্জন এবং ৭৩৭ কোটি ২৪ লাখ ২৬ হাজার ৫৮১ টাকার ‘অস্বাভাবিক ও সন্দেহজনক’ লেনদেনের প্রমাণ পাওয়ার কথা জানিয়েছে দুর্নীতি দমন কমিশন (দুদক)। বুধবার এই চারজনের বিরুদ্ধে আদালতে অভিযোগপত্র দাখিলের অনুমোদন দিয়েছে সংস্থাটি।
মামলার অপর তিন আসামি হলেন আনিসুল হকের ছোট ভাইয়ের স্ত্রী জেবুন্নেসা বেগম হক কলি, ভাগনে শেখ মো. ইফতেখারুল ইসলাম ও ‘কাজিন’ হিসেবে পরিচিত মোহাম্মদ ইকবাল। দুদক সচিব মো. সাইদুর রহমান খান এক সংবাদ সম্মেলনে বলেন, এই তিনজন আনিসুল হককে অবৈধ সম্পদ অর্জনে প্রত্যক্ষ ও পরোক্ষভাবে সহায়তা করেছেন।
দুদকের তদন্তে আনিসুল হকের গ্রহণযোগ্য আয় পাওয়া গেছে ১৩ কোটি ২৭ লাখ ৯৫ হাজার ২৩৮ টাকা। এর বিপরীতে পারিবারিক ব্যয় ধরা হয়েছে ৪ কোটি ৬৬ লাখ ৫৪ হাজার ২৯৮ টাকা। ব্যয় বাদে নিট সঞ্চয় দাঁড়ায় ৮ কোটি ৬১ লাখ ৪০ হাজার ৯৪০ টাকা। অন্যদিকে আনিসুল হক ও অপর তিন আসামির নামে মোট ১৮০ কোটি ৮৫ লাখ ৭৩ হাজার ৮৮১ টাকার সম্পদ থাকার তথ্য পেয়েছে দুদক। গ্রহণযোগ্য নিট আয় বাদ দিলে ১৭২ কোটি ২৪ লাখ ৩২ হাজার ৯৪১ টাকার সম্পদকে ‘জ্ঞাত আয়ের উৎসের সঙ্গে অসংগতিপূর্ণ’ বলেছে সংস্থাটি।
দুদকের ভাষ্য, ২০০০ সালের ৩০ জুন থেকে ২০২৬ সালের ১৯ জুলাই পর্যন্ত সময়ে এসব সম্পদ অর্জিত হয়েছে। আনিসুল হকের নিজের নামে ৯৬ কোটি ৮০ লাখ ৬৬ হাজার ৮৮০ টাকার স্থাবর-অস্থাবর সম্পদ রয়েছে। অপর তিনজনের নামে রয়েছে ৮৪ কোটি ৫ লাখ ৭ হাজার ১ টাকার সম্পদ।
২৩ ব্যাংক হিসাবে ৭৩৭ কোটি টাকার লেনদেন|আনিসুল হক ও তাঁর স্বার্থসংশ্লিষ্ট ব্যক্তি ও প্রতিষ্ঠানের ২৩টি ব্যাংক হিসাবে মোট ৩৭৫ কোটি ৫০ লাখ ৯৯ হাজার ৪৫২ টাকা জমা এবং ৩৬১ কোটি ৭৩ লাখ ২৭ হাজার ১২৯ টাকা উত্তোলনের তথ্য পেয়েছে দুদক। জমা ও উত্তোলন মিলিয়ে মোট লেনদেনের পরিমাণ ৭৩৭ কোটি ২৪ লাখ ২৬ হাজার ৫৮১ টাকা।
দুদকের সন্দেহ, ‘দুর্নীতি ও ঘুষের মাধ্যমে’ পাওয়া অর্থের অবৈধ উৎস গোপন, রূপান্তর ও হস্তান্তরের উদ্দেশ্যে এসব লেনদেন করা হয়েছে।
২০২৫ সালের ১ জানুয়ারি আনিসুল হকের বিরুদ্ধে মামলাটি করেন দুদকের উপপরিচালক মো. জাহাঙ্গীর আলম। এজাহারে ১৪৬ কোটি ১৯ লাখ টাকার অবৈধ সম্পদ এবং ২৯টি ব্যাংক হিসাবে প্রায় ৬৬৬ কোটি টাকার লেনদেনের অভিযোগ ছিল।
অভিযোগপত্রে দুর্নীতি দমন কমিশন আইন, দুর্নীতি প্রতিরোধ আইন ও মানি লন্ডারিং প্রতিরোধ আইনের বিভিন্ন ধারার পাশাপাশি অপরাধে সহায়তা বা প্ররোচনার দায়ে দণ্ডবিধির ১০৯ ধারাও যুক্ত করা হয়েছে।
'''
predicted_domain_2, predicted_sub_domain_2 = predict_domain_and_sub_domain(sample_text_2)

print(f"\nSample Text: {sample_text_2}")
print(f"Predicted Domain: {predicted_domain_2}")
print(f"Predicted Sub-Domain: {predicted_sub_domain_2}")